In [3]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report, confusion_matrix, roc_auc_score
import statsmodels.api as sm
import xgboost as xgb
from datetime import datetime, timedelta
import re

print("="*80)
print("COMPLETE ALTERNATIVE CREDIT SCORING PIPELINE")
print("6 Macro Features: 3 Digital + 3 Economic Indicators")
print("="*80)

# ============================================================================
# STEP 1: READ ALL DATASETS
# ============================================================================
print("\n" + "="*80)
print("STEP 1: READING DATASETS")
print("="*80)

customer_df = pd.read_csv('Customer_financial_profiles_ADVANCED_SYNTHETIC.csv')
upi_df = pd.read_csv('UPI-data.csv')
nfs_df = pd.read_csv('Digital-NFS-data.csv')
defaulters_df = pd.read_csv('wilful-defaulters-cibil.csv')
gsdp_df = pd.read_csv('gross-state-value-data.csv')
unemployment_df = pd.read_csv('unemployment-data.csv')
inflation_df = pd.read_csv('CPI-data.csv')

print(f"✓ Customer records: {len(customer_df)}")
print(f"✓ UPI records: {len(upi_df)}")
print(f"✓ NFS records: {len(nfs_df)}")
print(f"✓ Defaulters: {len(defaulters_df)}")
print(f"✓ GSDP: {len(gsdp_df)}")
print(f"✓ Unemployment: {len(unemployment_df)}")
print(f"✓ Inflation: {len(inflation_df)}")

# ============================================================================
# STEP 2: DATA CLEANING
# ============================================================================
print("\n" + "="*80)
print("STEP 2: DATA CLEANING")
print("="*80)

# Clean UPI volume column
if upi_df['Volume (in Mn)'].dtype == 'object':
    upi_df['Volume (in Mn)'] = upi_df['Volume (in Mn)'].str.replace(',', '').astype(float)
else:
    upi_df['Volume (in Mn)'] = upi_df['Volume (in Mn)'].astype(float)

# Standardize state names
customer_df['merchant_state'] = customer_df['merchant_state'].str.strip().str.title()
upi_df['State / Union Territory'] = upi_df['State / Union Territory'].str.strip().str.title()
nfs_df['state'] = nfs_df['state'].str.strip().str.title()
defaulters_df['State'] = defaulters_df['State'].str.strip().str.title()
gsdp_df['state'] = gsdp_df['state'].str.strip().str.title()
unemployment_df['state'] = unemployment_df['state'].str.strip().str.title()
inflation_df['state'] = inflation_df['state'].str.strip().str.title()

print("✓ Data cleaned and standardized")

# ============================================================================
# STEP 3: FISCAL YEAR CONVERSION HELPER
# ============================================================================
def extract_fiscal_year(fy_string):
    """Extract starting year from fiscal year string"""
    if pd.isna(fy_string):
        return None
    fy_string = str(fy_string).strip()
    match = re.search(r'(\d{4})', fy_string)
    if match:
        return int(match.group(1))
    return None

# ============================================================================
# STEP 4: CALCULATE ALL 6 MACRO FEATURES
# ============================================================================
print("\n" + "="*80)
print("STEP 3: CALCULATING 6 MACRO FEATURES")
print("="*80)

# State populations for per-capita calculations
state_populations = {
    'Uttar Pradesh': 199.8, 'Maharashtra': 112.4, 'Bihar': 104.1,
    'West Bengal': 91.3, 'Madhya Pradesh': 72.6, 'Tamil Nadu': 72.1,
    'Rajasthan': 68.5, 'Karnataka': 61.1, 'Gujarat': 60.4,
    'Andhra Pradesh': 49.5, 'Odisha': 42.0, 'Telangana': 35.2,
    'Kerala': 33.4, 'Jharkhand': 33.0, 'Assam': 31.2,
    'Punjab': 27.7, 'Chhattisgarh': 25.5, 'Haryana': 25.4,
    'Delhi': 16.8, 'Jammu And Kashmir': 12.5, 'Uttarakhand': 10.1,
    'Himachal Pradesh': 6.9, 'Tripura': 3.7, 'Meghalaya': 3.0,
    'Manipur': 2.9, 'Nagaland': 2.0, 'Goa': 1.5,
    'Arunachal Pradesh': 1.4, 'Puducherry': 1.2, 'Mizoram': 1.1,
    'Chandigarh': 1.1, 'Sikkim': 0.6, 'Andaman And Nicobar Islands': 0.4,
    'Andaman & Nicobar': 0.4, 'Dadra And Nagar Haveli': 0.3,
    'Daman And Diu': 0.2, 'Lakshadweep': 0.06
}

# --- MACRO 1: DVI (Digital Velocity Index) ---
print("\n1. DVI - Digital Velocity Index (Per-capita UPI adoption)")
upi_time_state = upi_df.groupby(['State / Union Territory', 'Year', 'Month'])['Volume (in Mn)'].mean().reset_index()
upi_time_state.columns = ['State', 'Year', 'Month', 'State_Volume']
upi_time_state['Population'] = upi_time_state['State'].map(state_populations).fillna(1)
upi_time_state['Volume_Per_Capita'] = upi_time_state['State_Volume'] / upi_time_state['Population']

national_population = sum(state_populations.values())
upi_national = upi_df.groupby(['Year', 'Month'])['Volume (in Mn)'].mean().reset_index()
upi_national['National_Per_Capita'] = upi_national['Volume (in Mn)'] / national_population
upi_time_state = upi_time_state.merge(upi_national[['Year', 'Month', 'National_Per_Capita']], on=['Year', 'Month'])
upi_time_state['DERIVED_UPI_DVI'] = upi_time_state['Volume_Per_Capita'] / upi_time_state['National_Per_Capita']
print(f"   ✓ Created: {len(upi_time_state)} state-month combinations")

# --- MACRO 2: IRS (Infrastructure Reliance Score) ---
print("\n2. IRS - Infrastructure Reliance Score (NFS banking access)")
month_map = {'January': 1, 'February': 2, 'March': 3, 'April': 4, 'May': 5, 'June': 6,
             'July': 7, 'August': 8, 'September': 9, 'October': 10, 'November': 11, 'December': 12}
nfs_df['Month_Num'] = nfs_df['month'].map(month_map)
nfs_time_state = nfs_df.groupby(['state', 'year', 'Month_Num'])['volume'].sum().reset_index()
nfs_time_state.columns = ['State', 'Year', 'Month', 'State_Volume']
nfs_national = nfs_df.groupby(['year', 'Month_Num'])['volume'].sum().reset_index()
nfs_national.columns = ['Year', 'Month', 'National_Volume']
nfs_time_state = nfs_time_state.merge(nfs_national, on=['Year', 'Month'])
nfs_time_state['DERIVED_NFS_IRS'] = nfs_time_state['State_Volume'] / nfs_time_state['National_Volume']
print(f"   ✓ Created: {len(nfs_time_state)} state-month combinations")

# --- MACRO 3: RCRES (Regional Credit Risk Environment Score) ---
print("\n3. RCRES - Regional Credit Risk Environment (Defaulter concentration)")
rcres_state = defaulters_df.groupby('State')['Outstanding Amount(in Lakhs)'].sum().reset_index()
national_default = defaulters_df['Outstanding Amount(in Lakhs)'].sum()
rcres_state['DERIVED_EXPERIAN_RCRES'] = rcres_state['Outstanding Amount(in Lakhs)'] / national_default
rcres_state.columns = ['State', 'Default_Amount', 'DERIVED_EXPERIAN_RCRES']
print(f"   ✓ Created: {len(rcres_state)} states")

# --- MACRO 4: Z_GSDPG (Economic Growth Score) ---
print("\n4. Z_GSDPG - Economic Growth (GSDP growth rate)")
gsdp_df['fiscal_year_num'] = gsdp_df['fiscal_year'].apply(extract_fiscal_year)
gsdp_filtered = gsdp_df[gsdp_df['fiscal_year_num'].isin([2022, 2023, 2024])].copy()
gsdp_filtered = gsdp_filtered.sort_values(['state', 'fiscal_year_num'])
gsdp_filtered['gsdp_lag'] = gsdp_filtered.groupby('state')['constant_prices'].shift(1)
gsdp_filtered['growth_rate'] = ((gsdp_filtered['constant_prices'] - gsdp_filtered['gsdp_lag']) / gsdp_filtered['gsdp_lag']) * 100
gsdp_filtered = gsdp_filtered[gsdp_filtered['growth_rate'].notna()]

gsdp_monthly = []
for _, row in gsdp_filtered.iterrows():
    for m in range(4, 13):
        gsdp_monthly.append({'State': row['state'], 'Year': row['fiscal_year_num'], 'Month': m, 'Z_GSDPG': row['growth_rate']})
    for m in range(1, 4):
        gsdp_monthly.append({'State': row['state'], 'Year': row['fiscal_year_num']+1, 'Month': m, 'Z_GSDPG': row['growth_rate']})
gsdp_time_state = pd.DataFrame(gsdp_monthly)
print(f"   ✓ Created: {len(gsdp_time_state)} state-month combinations")

# --- MACRO 5: Z_UNEM (Labor Market Stress Score) ---
print("\n5. Z_UNEM - Labor Market Stress (Unemployment rate)")
unemployment_df['fiscal_year_num'] = unemployment_df['fiscal_year'].apply(extract_fiscal_year)
unem_filtered = unemployment_df[unemployment_df['fiscal_year_num'].isin([2022, 2023])].copy()
unem_agg = unem_filtered.groupby(['state', 'fiscal_year_num'])['unemployment_rate'].mean().reset_index()

unem_monthly = []
for _, row in unem_agg.iterrows():
    for m in range(4, 13):
        unem_monthly.append({'State': row['state'], 'Year': row['fiscal_year_num'], 'Month': m, 'Z_UNEM': row['unemployment_rate']})
    for m in range(1, 4):
        unem_monthly.append({'State': row['state'], 'Year': row['fiscal_year_num']+1, 'Month': m, 'Z_UNEM': row['unemployment_rate']})
unem_time_state = pd.DataFrame(unem_monthly)
print(f"   ✓ Created: {len(unem_time_state)} state-month combinations")

# --- MACRO 6: Z_CPII (Inflation Exposure Score) ---
print("\n6. Z_CPII - Inflation Exposure (CPI inflation)")
inflation_df['fiscal_year_num'] = inflation_df['fiscal_year'].apply(extract_fiscal_year)
infl_filtered = inflation_df[inflation_df['fiscal_year_num'].isin([2022, 2023, 2024])].copy()
infl_general = infl_filtered[infl_filtered['category'].str.contains('General', case=False, na=False)]
infl_agg = infl_general.groupby(['state', 'fiscal_year_num'])['percentage'].mean().reset_index()

infl_monthly = []
for _, row in infl_agg.iterrows():
    for m in range(4, 13):
        infl_monthly.append({'State': row['state'], 'Year': row['fiscal_year_num'], 'Month': m, 'Z_CPII': row['percentage']})
    for m in range(1, 4):
        infl_monthly.append({'State': row['state'], 'Year': row['fiscal_year_num']+1, 'Month': m, 'Z_CPII': row['percentage']})
infl_time_state = pd.DataFrame(infl_monthly)
print(f"   ✓ Created: {len(infl_time_state)} state-month combinations")

# ============================================================================
# STEP 5: MERGE MACRO FEATURES WITH CUSTOMER DATA
# ============================================================================
print("\n" + "="*80)
print("STEP 4: MERGING MACRO FEATURES WITH CUSTOMER DATA")
print("="*80)

# Extract year-month from customer transactions
customer_df['transaction_date'] = pd.to_datetime(customer_df['date'])
customer_df['Year'] = customer_df['transaction_date'].dt.year
customer_df['Month'] = customer_df['transaction_date'].dt.month

# Merge all 6 macro features
print("\nMerging DVI...")
customer_df = customer_df.merge(upi_time_state[['State', 'Year', 'Month', 'DERIVED_UPI_DVI']], 
                                  left_on=['merchant_state', 'Year', 'Month'], 
                                  right_on=['State', 'Year', 'Month'], how='left')
print(f"  ✓ Matched: {customer_df['DERIVED_UPI_DVI'].notna().sum()}/{len(customer_df)}")

print("Merging IRS...")
customer_df = customer_df.merge(nfs_time_state[['State', 'Year', 'Month', 'DERIVED_NFS_IRS']], 
                                  left_on=['merchant_state', 'Year', 'Month'], 
                                  right_on=['State', 'Year', 'Month'], how='left', suffixes=('', '_nfs'))
print(f"  ✓ Matched: {customer_df['DERIVED_NFS_IRS'].notna().sum()}/{len(customer_df)}")

print("Merging RCRES...")
customer_df = customer_df.merge(rcres_state[['State', 'DERIVED_EXPERIAN_RCRES']], 
                                  left_on='merchant_state', right_on='State', how='left', suffixes=('', '_rcres'))
print(f"  ✓ Matched: {customer_df['DERIVED_EXPERIAN_RCRES'].notna().sum()}/{len(customer_df)}")

print("Merging Z_GSDPG...")
customer_df = customer_df.merge(gsdp_time_state, left_on=['merchant_state', 'Year', 'Month'], 
                                  right_on=['State', 'Year', 'Month'], how='left', suffixes=('', '_gsdp'))
print(f"  ✓ Matched: {customer_df['Z_GSDPG'].notna().sum()}/{len(customer_df)}")

print("Merging Z_UNEM...")
customer_df = customer_df.merge(unem_time_state, left_on=['merchant_state', 'Year', 'Month'], 
                                  right_on=['State', 'Year', 'Month'], how='left', suffixes=('', '_unem'))
print(f"  ✓ Matched: {customer_df['Z_UNEM'].notna().sum()}/{len(customer_df)}")

print("Merging Z_CPII...")
customer_df = customer_df.merge(infl_time_state, left_on=['merchant_state', 'Year', 'Month'], 
                                  right_on=['State', 'Year', 'Month'], how='left', suffixes=('', '_cpii'))
print(f"  ✓ Matched: {customer_df['Z_CPII'].notna().sum()}/{len(customer_df)}")

# Clean up duplicate State columns
state_cols = [c for c in customer_df.columns if c.startswith('State') and c != 'State']
customer_df = customer_df.drop(columns=state_cols, errors='ignore')

# ============================================================================
# STEP 6: CREATE TARGET & FEATURES
# ============================================================================
print("\n" + "="*80)
print("STEP 5: FEATURE ENGINEERING")
print("="*80)

# Target variable
customer_df['Y_Default_Risk'] = (customer_df['credit_score'] < 650).astype(int)
print(f"\nTarget variable created:")
print(f"  High Risk (Y=1): {customer_df['Y_Default_Risk'].sum()}")
print(f"  Low Risk (Y=0): {(customer_df['Y_Default_Risk']==0).sum()}")

# Additional features
customer_df['Debt_to_Income_Ratio'] = customer_df['total_debt'] / customer_df['yearly_income']

# Drop duplicates (one record per customer)
modeling_data = customer_df.drop_duplicates(subset='id', keep='first').copy()
print(f"\nUnique customers: {len(modeling_data)}")

# Fill missing macro features
for col in ['DERIVED_UPI_DVI', 'DERIVED_NFS_IRS', 'DERIVED_EXPERIAN_RCRES']:
    modeling_data[col] = modeling_data[col].fillna(0)

for col in ['Z_GSDPG', 'Z_UNEM', 'Z_CPII']:
    mean_val = modeling_data[col].mean()
    modeling_data[col] = modeling_data[col].fillna(mean_val)
    print(f"  Filled {col} with mean: {mean_val:.2f}")

# Encode gender
le = LabelEncoder()
modeling_data['gender_encoded'] = le.fit_transform(modeling_data['gender'])

# ============================================================================
# STEP 7: DEFINE FEATURES FOR MODELING
# ============================================================================
print("\n" + "="*80)
print("STEP 6: DEFINING FEATURE SETS")
print("="*80)

micro_features = ['current_age', 'gender_encoded', 'num_credit_cards',
                  'yearly_income', 'total_debt', 'Debt_to_Income_Ratio']

macro_features = ['DERIVED_UPI_DVI', 'DERIVED_NFS_IRS', 'DERIVED_EXPERIAN_RCRES',
                  'Z_GSDPG', 'Z_UNEM', 'Z_CPII']

all_features = micro_features + macro_features

print(f"\n✓ Micro features: {len(micro_features)}")
print(f"✓ Macro features: {len(macro_features)}")
for f in macro_features:
    print(f"    • {f}")
print(f"✓ Total features: {len(all_features)}")

# Prepare X and y
X = modeling_data[all_features].copy()
y = modeling_data['Y_Default_Risk'].copy()

X = X.replace([np.inf, -np.inf], np.nan).fillna(X.median())

# ============================================================================
# STEP 8: TRAIN-TEST SPLIT & SCALING
# ============================================================================
print("\n" + "="*80)
print("STEP 7: TRAIN-TEST SPLIT")
print("="*80)

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)
print(f"✓ Train: {len(X_train)}, Test: {len(X_test)}")

scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

# ============================================================================
# STEP 9: LOGISTIC REGRESSION WITH SIGNIFICANCE TESTING
# ============================================================================
print("\n" + "="*80)
print("STEP 8: LOGISTIC REGRESSION - STATISTICAL SIGNIFICANCE")
print("="*80)

X_train_const = sm.add_constant(X_train_scaled)

sm_model = sm.Logit(y_train, X_train_const).fit(method='newton', maxiter=1000, disp=0)

results_df = pd.DataFrame({
    'Feature': ['Intercept'] + all_features,
    'Coefficient': sm_model.params.values,
    'Std_Error': sm_model.bse.values,
    'z_value': sm_model.tvalues.values,
    'P_value': sm_model.pvalues.values,
    'Sig': ['***' if p<0.001 else '**' if p<0.01 else '*' if p<0.05 else '†' if p<0.10 else 'ns' 
            for p in sm_model.pvalues.values]
})

print(f"\n✓ Model converged (Pseudo R²: {sm_model.prsquared:.4f})")

print("\n" + "="*80)
print("ALL FEATURES - SIGNIFICANCE TABLE")
print("="*80)
print("Significance: *** p<0.001, ** p<0.01, * p<0.05, † p<0.10, ns not significant\n")
print(results_df.to_string(index=False))

# Display macro features separately
print("\n" + "="*80)
print("MACRO FEATURES ANALYSIS")
print("="*80)

macro_results = results_df[results_df['Feature'].isin(macro_features)]

print("\n📊 DIGITAL INFRASTRUCTURE INDICATORS:")
print("-"*80)
digital = macro_results[macro_results['Feature'].str.contains('DERIVED')]
print(digital[['Feature', 'Coefficient', 'P_value', 'Sig']].to_string(index=False))

print("\n📈 ECONOMIC INDICATORS:")
print("-"*80)
economic = macro_results[macro_results['Feature'].str.contains('Z_')]
print(economic[['Feature', 'Coefficient', 'P_value', 'Sig']].to_string(index=False))

print("\n" + "="*80)
print("MACRO SUMMARY")
print("="*80)
print(f"Highly significant (p<0.001): {(macro_results['P_value'] < 0.001).sum()}/6")
print(f"Very significant (p<0.01): {((macro_results['P_value'] >= 0.001) & (macro_results['P_value'] < 0.01)).sum()}/6")
print(f"Significant (p<0.05): {((macro_results['P_value'] >= 0.01) & (macro_results['P_value'] < 0.05)).sum()}/6")
print(f"Marginally significant (p<0.10): {((macro_results['P_value'] >= 0.05) & (macro_results['P_value'] < 0.10)).sum()}/6")
print(f"Not significant: {(macro_results['P_value'] >= 0.10).sum()}/6")

# ============================================================================
# STEP 10: MODEL PERFORMANCE
# ============================================================================
print("\n" + "="*80)
print("STEP 9: MODEL PERFORMANCE")
print("="*80)

lr_sklearn = LogisticRegression(max_iter=1000, random_state=42)
lr_sklearn.fit(X_train_scaled, y_train)
y_pred_proba = lr_sklearn.predict_proba(X_test_scaled)[:, 1]
auc_lr = roc_auc_score(y_test, y_pred_proba)

print(f"\nLogistic Regression ROC-AUC: {auc_lr:.4f}")
print(f"Classification Report:")
print(classification_report(y_test, lr_sklearn.predict(X_test_scaled), target_names=['Low Risk', 'High Risk']))

# ============================================================================
# STEP 11: XGBOOST
# ============================================================================
print("\n" + "="*80)
print("STEP 10: XGBOOST - FEATURE IMPORTANCE")
print("="*80)

dtrain = xgb.DMatrix(X_train, label=y_train, feature_names=all_features)
dtest = xgb.DMatrix(X_test, label=y_test, feature_names=all_features)

xgb_model = xgb.train({'objective': 'binary:logistic', 'eval_metric': 'auc', 'max_depth': 6,
                       'learning_rate': 0.1, 'subsample': 0.8, 'colsample_bytree': 0.8, 'seed': 42},
                      dtrain, num_boost_round=100, evals=[(dtest, 'test')], 
                      early_stopping_rounds=10, verbose_eval=False)

auc_xgb = roc_auc_score(y_test, xgb_model.predict(dtest))
print(f"\nXGBoost ROC-AUC: {auc_xgb:.4f}")

importance = pd.DataFrame({'Feature': list(xgb_model.get_score(importance_type='gain').keys()),
                           'Importance': list(xgb_model.get_score(importance_type='gain').values())}).sort_values('Importance', ascending=False)
print("\nTop 10 Features:")
print(importance.head(10).to_string(index=False))

print("\nMacro Features in XGBoost:")
print(importance[importance['Feature'].isin(macro_features)].to_string(index=False))

# ============================================================================
# FINAL SUMMARY
# ============================================================================
print("\n" + "="*80)
print("FINAL SUMMARY")
print("="*80)
print(f"\n✓ Pipeline completed successfully")
print(f"✓ All 6 macro features included: {len(macro_features)}")
print(f"✓ Logistic Regression AUC: {auc_lr:.4f}")
print(f"✓ XGBoost AUC: {auc_xgb:.4f}")
print(f"✓ Improvement: {((auc_xgb - auc_lr) / auc_lr * 100):.2f}%")

# Export
results_df.to_csv('complete_significance_results.csv', index=False)
macro_results.to_csv('macro_features_final.csv', index=False)
modeling_data.to_csv('modeling_dataset_complete.csv', index=False)

print(f"\n✓ Files exported:")
print(f"  - complete_significance_results.csv")
print(f"  - macro_features_final.csv")
print(f"  - modeling_dataset_complete.csv")

print("\n" + "="*80)
print("ANALYSIS COMPLETE!")
print("="*80)

COMPLETE ALTERNATIVE CREDIT SCORING PIPELINE
6 Macro Features: 3 Digital + 3 Economic Indicators

STEP 1: READING DATASETS
✓ Customer records: 50000
✓ UPI records: 1221
✓ NFS records: 10375
✓ Defaulters: 1501692
✓ GSDP: 470
✓ Unemployment: 2736
✓ Inflation: 1584

STEP 2: DATA CLEANING
✓ Data cleaned and standardized

STEP 3: CALCULATING 6 MACRO FEATURES

1. DVI - Digital Velocity Index (Per-capita UPI adoption)
   ✓ Created: 1184 state-month combinations

2. IRS - Infrastructure Reliance Score (NFS banking access)
   ✓ Created: 2096 state-month combinations

3. RCRES - Regional Credit Risk Environment (Defaulter concentration)
   ✓ Created: 44 states

4. Z_GSDPG - Economic Growth (GSDP growth rate)
   ✓ Created: 672 state-month combinations

5. Z_UNEM - Labor Market Stress (Unemployment rate)
   ✓ Created: 912 state-month combinations

6. Z_CPII - Inflation Exposure (CPI inflation)
   ✓ Created: 1296 state-month combinations

STEP 4: MERGING MACRO FEATURES WITH CUSTOMER DATA

Merging D